# 🦅 Bird Detection — YOLOv8s Training
**Dataset:** kakitamedia/drone_dataset (47,260 images · 3 classes: hawk, crow, wild_bird)

**Goal:** Train YOLOv8s at 1280px to detect small distant birds for shrimp pond protection

**Platform:** Kaggle (free T4 GPU) or Google Colab Pro

## Step 1 — Install dependencies

In [ ]:
!pip install ultralytics sahi gdown tqdm --quiet
import ultralytics
ultralytics.checks()

## Step 2 — Download dataset from Google Drive

In [ ]:
import gdown, zipfile, os
from pathlib import Path

# Annotation file (single JSON ~50MB)
ANNOT_ID = '1P-yM34AjsRXFDyOzGW7MbJnpuP3f3IKy'
gdown.download(id=ANNOT_ID, output='annotations.json', quiet=False)

# Images folder (~67GB split zips) — download from:
# https://drive.google.com/drive/folders/11H30-Oh_Ybi_LzsRot2soHaNp2ZWlt4i
# On Kaggle: add the dataset as a Kaggle dataset input instead
print('Annotation downloaded!')
print('NOTE: Download image zips manually from Google Drive and upload to Kaggle dataset')

## Step 3 — Convert JSON annotations → YOLO format

In [ ]:
import json
import shutil
from pathlib import Path
from tqdm import tqdm

IMAGES_ROOT = Path('/kaggle/input/drone-bird-dataset/images')  # adjust to your path
ANNOT_FILE  = Path('annotations.json')
OUTPUT_DIR  = Path('dataset')
IMAGE_W, IMAGE_H = 3840, 2160

CLASS_MAP = {'hawk': 0, 'crow': 1, 'wild bird': 2, 'wild_bird': 2}

def convert_split(entries, split, img_root):
    out_img = OUTPUT_DIR / 'images' / split
    out_lbl = OUTPUT_DIR / 'labels' / split
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    converted, skipped = 0, 0
    for entry in tqdm(entries, desc=split):
        img_src = img_root / entry['path']
        if not img_src.exists():
            skipped += 1
            continue
        lines = []
        for bbox, label in zip(entry.get('bbox', []), entry.get('label', [])):
            cls = CLASS_MAP.get(label.lower().strip())
            if cls is None: continue
            x, y, w, h = bbox
            cx = max(0.0, min(1.0, (x + w/2) / IMAGE_W))
            cy = max(0.0, min(1.0, (y + h/2) / IMAGE_H))
            nw = max(0.0, min(1.0, w / IMAGE_W))
            nh = max(0.0, min(1.0, h / IMAGE_H))
            lines.append(f'{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        if not lines:
            skipped += 1
            continue
        shutil.copy2(img_src, out_img / img_src.name)
        (out_lbl / (img_src.stem + '.txt')).write_text('\n'.join(lines))
        converted += 1
    print(f'  {split}: converted={converted}, skipped={skipped}')

with open(ANNOT_FILE) as f:
    all_data = json.load(f)

# Split 90/10 if no pre-split, otherwise the dataset has train/val JSON separately
split_idx = int(len(all_data) * 0.9)
train_data = all_data[:split_idx]
val_data   = all_data[split_idx:]

convert_split(train_data, 'train', IMAGES_ROOT)
convert_split(val_data,   'val',   IMAGES_ROOT)

# Write dataset YAML
yaml = f"""path: {OUTPUT_DIR.resolve()}
train: images/train
val:   images/val
nc: 3
names:
  0: hawk
  1: crow
  2: wild_bird
"""
(OUTPUT_DIR / 'dataset.yaml').write_text(yaml)
print('dataset.yaml written!')

## Step 4 — Train YOLOv8s

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # pretrained COCO weights as starting point

results = model.train(
    data    = 'dataset/dataset.yaml',
    epochs  = 50,
    imgsz   = 1280,      # large input = better small/distant bird detection
    batch   = 8,         # reduce to 4 if GPU OOM
    device  = 0,         # GPU
    workers = 4,
    patience= 15,        # early stop if no improvement
    save    = True,
    project = 'bird_runs',
    name    = 'yolov8s_birds_1280',
    # Augmentation tuned for aerial/distant small objects
    hsv_h   = 0.015,
    hsv_s   = 0.5,
    hsv_v   = 0.3,
    flipud  = 0.5,       # birds seen from above — vertical flip OK
    fliplr  = 0.5,
    mosaic  = 1.0,
    scale   = 0.5,
    translate = 0.1,
)

## Step 5 — Validate and check metrics

In [ ]:
# Load best weights
best_model = YOLO('bird_runs/yolov8s_birds_1280/weights/best.pt')
metrics = best_model.val(data='dataset/dataset.yaml', imgsz=1280)

print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.p.mean():.4f}')
print(f'Recall   : {metrics.box.r.mean():.4f}')

## Step 6 — Test on a sample image with SAHI (sliced inference for tiny birds)

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import visualize_object_predictions
import matplotlib.pyplot as plt

detection_model = AutoDetectionModel.from_pretrained(
    model_type   = 'ultralytics',
    model_path   = 'bird_runs/yolov8s_birds_1280/weights/best.pt',
    confidence_threshold = 0.3,
    device       = 'cuda:0',
)

# Pick any val image to test
import glob
test_img = glob.glob('dataset/images/val/*.jpg')[0]
print(f'Testing on: {test_img}')

result = get_sliced_prediction(
    test_img,
    detection_model,
    slice_height         = 640,
    slice_width          = 640,
    overlap_height_ratio = 0.2,
    overlap_width_ratio  = 0.2,
)

print(f'Detections: {len(result.object_prediction_list)}')
result.export_visuals(export_dir='sahi_output/')
img = plt.imread('sahi_output/prediction_visual.png')
plt.figure(figsize=(16,9))
plt.imshow(img)
plt.axis('off')
plt.title('SAHI Sliced Inference — Distant Bird Detection')
plt.show()

## Step 7 — Export to TensorRT for Jetson Orin Nano Super

In [ ]:
# NOTE: Run this step ON the Jetson, not on Kaggle/Colab
# Copy best.pt to Jetson first, then run:

# from ultralytics import YOLO
# model = YOLO('best.pt')
# model.export(
#     format   = 'engine',   # TensorRT
#     imgsz    = 1280,
#     half     = True,       # FP16 — 2x faster on Jetson
#     device   = 0,
# )
# # This produces best.engine — use this for real-time inference on Jetson

print('Export to TensorRT must be run ON the Jetson Orin Nano Super')
print('Copy best.pt to Jetson, then run the commented code above')

## Step 8 — Download trained weights
Download `best.pt` from `bird_runs/yolov8s_birds_1280/weights/best.pt`

In [ ]:
# On Kaggle: use the output panel to download
# On Colab:
from google.colab import files
files.download('bird_runs/yolov8s_birds_1280/weights/best.pt')